# Occlusion benchmark: SMIRK_em1 (baseline) or a trained model

Evaluates a SMIRK encoder on all clean test faces, clean and with synthetic occlusions
(`src/evaluation/occlusion_benchmark.py`: random parts, mouth / eyes / eyebrows / nose, lower / upper face), and on the
**real occlusions** (glasses, hats, hands) of `eval-extras`. Every score has a 95% bootstrap interval; section 7 compares
two models on the same faces. No rendering is needed, so PyTorch3D is not installed.

Before running:
- Settings: **Accelerator GPU T4 x1**, **Internet on**.
- Inputs: `flame2020`, `clean-faces`, `clean-faces-rafdb`, `expw-crops`, `affectnet_1`, `rafdb-crops`, `eval-extras`
  (real occlusions + duplicate list: the emotion probe is trained without the train faces that duplicate test faces).
- To evaluate a trained model: add the output of its training run (notebook 06) as input and set `RUN` in section 3.

Output `/kaggle/working/eval_<model>/`: `summary.csv`, `per_image.csv`, `features.npz`, `config.json`.
Keep the notebook and outputs **Private** (FLAME / SMIRK licenses).

## 1. Environment and code

In [ ]:
import sys, os, glob, subprocess
import torch
print('python', sys.version.split()[0], '| torch', torch.__version__, '| GPU', torch.cuda.is_available())
assert torch.cuda.is_available(), 'enable the GPU accelerator'

os.chdir('/kaggle/working')  # leave /tmp/smirk before deleting it (a later cell changes into it)
!rm -rf /tmp/smirk && git clone -q --depth 1 -b occlusion-aware https://github.com/THUONGVIP09/smirk.git /tmp/smirk
!cd /tmp/smirk && git log --oneline -1
!pip install -q timm==0.9.16 gdown

## 2. FLAME 2020 and SMIRK_em1.pt

In [ ]:
import shutil
from kaggle_secrets import UserSecretsClient

assert os.path.isdir('/tmp/smirk/assets'), 'the SMIRK code is missing in /tmp/smirk: run the cell that clones the repository first'
os.makedirs('/tmp/smirk/assets/FLAME2020', exist_ok=True)

import requests, time, zipfile

# kept in /tmp (not in the notebook output: FLAME may not be redistributed), so that re-running this cell
# in the same session does not download it again
FLAME_ZIP = '/tmp/FLAME2020.zip'
# a private Kaggle dataset with FLAME (the zip, or the extracted generic_model.pkl) avoids downloading it every run
local = [p for p in glob.glob('/kaggle/input/**/*', recursive=True)
         if os.path.basename(p) == 'generic_model.pkl' or (p.lower().endswith('.zip') and 'flame' in os.path.basename(p).lower())]
if local and local[0].endswith('.pkl'):
    shutil.copytree(os.path.dirname(local[0]), '/tmp/smirk/assets/FLAME2020', dirs_exist_ok=True)
    print('FLAME from input:', os.path.dirname(local[0]))
elif local:
    FLAME_ZIP = local[0]
    print('FLAME zip from input:', FLAME_ZIP)
if not os.path.exists('/tmp/smirk/assets/FLAME2020/generic_model.pkl') and not (os.path.exists(FLAME_ZIP) and zipfile.is_zipfile(FLAME_ZIP)):
    secrets = UserSecretsClient()
    credentials = {'username': secrets.get_secret('FLAME_USERNAME'), 'password': secrets.get_secret('FLAME_PASSWORD')}
    url = 'https://download.is.tue.mpg.de/download.php?domain=flame&sfile=FLAME2020.zip&resume=1'
    for attempt in range(3):  # the credentials are sent in the request body only, never printed
        try:
            with requests.post(url, data=credentials, stream=True, timeout=600) as response:
                status = response.status_code
                if status == 200:
                    with open(FLAME_ZIP, 'wb') as f:
                        for chunk in response.iter_content(1 << 20):
                            f.write(chunk)
        except requests.RequestException as e:
            status = type(e).__name__
        if os.path.exists(FLAME_ZIP) and zipfile.is_zipfile(FLAME_ZIP):
            break
        print(f'FLAME download attempt {attempt + 1} failed (HTTP status {status}), retrying in 30 s')
        time.sleep(30)
    del credentials
    assert zipfile.is_zipfile(FLAME_ZIP), ('FLAME download failed: HTTP 401/403 = wrong FLAME_USERNAME / FLAME_PASSWORD '
                                           'or account not activated; 429/5xx = server busy, try again later')
if not os.path.exists('/tmp/smirk/assets/FLAME2020/generic_model.pkl'):
    assert subprocess.run(['unzip', '-q', '-o', FLAME_ZIP, '-d', '/tmp/smirk/assets/FLAME2020/']).returncode == 0, \
        'could not extract the FLAME zip'
found = glob.glob('/tmp/smirk/assets/FLAME2020/**/generic_model.pkl', recursive=True)
assert found, 'generic_model.pkl is not in the FLAME zip'
if os.path.dirname(found[0]) != '/tmp/smirk/assets/FLAME2020':  # the zip may contain a FLAME2020/ folder
    for name in os.listdir(os.path.dirname(found[0])):
        shutil.move(os.path.join(os.path.dirname(found[0]), name), '/tmp/smirk/assets/FLAME2020/')

!mkdir -p /tmp/smirk/pretrained_models
!gdown -q 1T65uEd9dVLHgVw5KiUYL66NUee-MCzoE -O /tmp/smirk/pretrained_models/SMIRK_em1.pt
!ls -la /tmp/smirk/assets/FLAME2020/generic_model.pkl /tmp/smirk/pretrained_models/SMIRK_em1.pt

## 3. Find the inputs

In [ ]:
CLEAN_DIRS = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/clean_meta.csv', recursive=True))
IMAGE_DIRS = {}
for folder in glob.glob('/kaggle/input/**/images', recursive=True):
    if not os.path.isdir(folder):
        continue
    # dataset of an image folder, from the keyword in its path (works for rafdb_crops, rafdb-crops, affectnet_1, ...)
    path = os.path.relpath(folder, '/kaggle/input').lower()
    name = next((k for k in ('expw', 'affectnet', 'rafdb') if k in path), None)
    if name:
        IMAGE_DIRS[name] = folder
        print(f'{name}: {folder} ({len(os.listdir(folder))} images)')
print('clean dirs:', CLEAN_DIRS)
print('image dirs:', IMAGE_DIRS)
if not IMAGE_DIRS:
    !find /kaggle/input -maxdepth 6 -type d | head -40
assert CLEAN_DIRS and IMAGE_DIRS, 'add clean-faces and the *-crops datasets as input'

# eval-extras (notebook 02/kaggle_duplicates_real_occlusion): real occlusions and the train faces that duplicate test faces
REAL_DIRS = sorted(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/real_occluded_meta.csv', recursive=True))
EXCLUDE = sorted(glob.glob('/kaggle/input/**/exclude_train.csv', recursive=True))
print('real occlusions:', REAL_DIRS or 'none', '| duplicate list:', EXCLUDE or 'none')
if not (REAL_DIRS and EXCLUDE):
    print('WARNING: add the eval-extras dataset: without it there are no real occlusions and the probe may train on duplicates of test faces')

# the model to evaluate: '' = SMIRK_em1 (baseline), or the RUN name of a training run of notebook 06 given as input
RUN = ''   # e.g. 'original_transfer', 'occlusion_aware_finetune'
if RUN:
    found = [p for n in ['model_final.pt', 'model_last.pt'] for p in glob.glob(f'/kaggle/input/**/logs/{RUN}/{n}', recursive=True)]
    assert found, f'no logs/{RUN}/model_final.pt in the inputs: add the output of that training run'
    CHECKPOINT = found[0]
else:
    CHECKPOINT = 'pretrained_models/SMIRK_em1.pt'
NAME = RUN or 'smirk_em1'
print('model:', CHECKPOINT)

ARGS = ' '.join([f'--clean_dir "{d}"' for d in CLEAN_DIRS] + [f'--image_dir "{k}={v}"' for k, v in IMAGE_DIRS.items()] +
                [f'--real_dir "{d}"' for d in REAL_DIRS] + [f'--exclude "{e}"' for e in EXCLUDE])
os.chdir('/tmp/smirk')  # FLAME assets are loaded with paths relative to the repository root

## 4. Quick check on 200 test faces (a few minutes)

In [ ]:
!python -m src.evaluation.occlusion_benchmark {ARGS} --checkpoint {CHECKPOINT} --out_dir /tmp/quick_check --limit 200 --probe_train_max 3000 --bootstrap 200

## 5. Full evaluation on all test faces

In [ ]:
OUT = f'/kaggle/working/eval_{NAME}'
!python -m src.evaluation.occlusion_benchmark {ARGS} --checkpoint {CHECKPOINT} --out_dir {OUT}

## 6. Results

`expr_mm`: how far (mm, mean over the FLAME vertices) the occluded-image expression mesh is from the clean one.
`exp_scale`: ||psi_occluded|| / ||psi_clean||, > 1 = more intense expressions under occlusion (SMIRK paper, Sec. 4.5).
`emotion_mean_class_acc`: MLP emotion probe, accuracy averaged over the classes (8 for AffectNet, 7 for ExpW / RAF-DB).
The SMIRK paper reports 0.64 (8 classes) on the official AffectNet test set; our AffectNet split differs.
`*_ci_low` / `*_ci_high`: 95% bootstrap interval; `*_change_ci_*`: interval of the change w.r.t. clean
(it does not contain 0 = the drop is significant).

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

summary = pd.read_csv(f'{OUT}/summary.csv')
pd.set_option('display.width', 200)
print(summary.round(3).to_string(index=False))

# main score with its interval: accuracy (%) [95% interval], change w.r.t. clean [95% interval]
def with_ci(row, m):
    text = f"{100 * row[m]:.1f} [{100 * row[m + '_ci_low']:.1f}, {100 * row[m + '_ci_high']:.1f}]"
    if pd.notna(row.get(m + '_change_ci_low')):
        text += f"  {100 * row[m + '_change']:+.1f} [{100 * row[m + '_change_ci_low']:+.1f}, {100 * row[m + '_change_ci_high']:+.1f}]"
    return text
table = summary.assign(mean_class_acc=summary.apply(lambda r: with_ci(r, 'emotion_mean_class_acc'), axis=1))
print(table.pivot(index='protocol', columns='dataset', values='mean_class_acc').reindex(summary['protocol'].unique()).to_string())

overall = summary[(summary['dataset'] == 'all') & ~summary['protocol'].str.startswith('real_')].set_index('protocol')
BLUE, INK, MUTED = '#2a78d6', '#0b0b0b', '#52514e'

def bars(ax, values, title, fmt):
    ax.bar(values.index, values.values, color=BLUE, width=0.6)
    for x, v in zip(values.index, values.values):
        ax.text(x, v, fmt.format(v), ha='center', va='bottom', fontsize=9, color=INK)
    ax.set_title(title, loc='left', fontsize=11, color=INK)
    ax.spines[['top', 'right']].set_visible(False)
    ax.tick_params(colors=MUTED)
    ax.grid(axis='y', color='#e5e5e3', linewidth=0.8); ax.set_axisbelow(True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
bars(axes[0], overall['expr_mm'].drop('clean'), 'Expression error vs clean image (mm)', '{:.2f}')
bars(axes[1], overall['exp_scale'].drop('clean'), 'Expression intensity, occluded / clean', '{:.2f}')
axes[1].axhline(1, color=MUTED, linewidth=1, linestyle='--')
bars(axes[2], overall['emotion_mean_class_acc'] * 100, 'Emotion probe, mean class accuracy (%)', '{:.1f}')
plt.tight_layout(); plt.show()

## 6b. Are the parts that are NOT occluded affected?

`expr_mm_visible`: error on the face vertices that are not under the occluders (0 = the occlusion does not change
the visible parts). Per region: how much each face region moves when another part is occluded.

In [ ]:
import numpy as np
occ = overall.drop('clean')
print(occ[['occluded_area', 'expr_mm_visible', 'expr_mm_occluded', 'lmk_px_visible', 'lmk_px_occluded']].round(3))
print(occ[[f'expr_mm_{r}' for r in ['eyes', 'forehead', 'nose', 'mouth', 'cheeks']]].round(3))

ORANGE = '#eb6834'
x = np.arange(len(occ)); width = 0.38
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width / 2 - 0.01, occ['expr_mm_visible'], width, color=BLUE, label='visible parts')
ax.bar(x + width / 2 + 0.01, occ['expr_mm_occluded'], width, color=ORANGE, label='occluded parts')
for xi, v in zip(x, occ['expr_mm_visible']): ax.text(xi - width / 2, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9, color=INK)
for xi, v in zip(x, occ['expr_mm_occluded']): ax.text(xi + width / 2, v, f'{v:.2f}', ha='center', va='bottom', fontsize=9, color=INK)
ax.set_xticks(x, occ.index); ax.legend(frameon=False)
ax.set_title('Expression error vs clean image (mm), visible vs occluded face parts', loc='left', fontsize=11, color=INK)
ax.spines[['top', 'right']].set_visible(False); ax.grid(axis='y', color='#e5e5e3', linewidth=0.8); ax.set_axisbelow(True)
plt.tight_layout(); plt.show()


## 6c. Real occlusions (glasses, hats, hands / objects)

Emotion probe on faces with real occlusions (other faces than the clean test faces: the change is w.r.t. the clean test
faces of the same dataset, with independent intervals).

In [ ]:
real = summary[summary['protocol'].str.startswith('real_')]
if len(real):
    print(real[['dataset', 'protocol', 'faces', 'emotion_mean_class_acc', 'emotion_mean_class_acc_ci_low',
                'emotion_mean_class_acc_ci_high', 'emotion_mean_class_acc_change', 'emotion_mean_class_acc_change_ci_low',
                'emotion_mean_class_acc_change_ci_high']].round(3).to_string(index=False))
else:
    print('no real occlusions: add the eval-extras dataset as input')

## 7. Compare with another model (paired bootstrap on the same faces)

Add the output of another evaluation (an earlier run of this notebook, e.g. `eval_smirk_em1/`) as input and set
`REFERENCE` to its folder name. `B better` / `B worse` = significant at the 5% level, B = the model evaluated above.
Both evaluations must use the same inputs and options (same test faces, `--limit`, protocols, eval-extras).

In [ ]:
REFERENCE = 'eval_smirk_em1'   # folder of the reference evaluation among the inputs
found = [os.path.dirname(p) for p in glob.glob(f'/kaggle/input/**/{REFERENCE}/per_image.csv', recursive=True)]
if found and os.path.abspath(found[0]) != os.path.abspath(OUT):
    !python -m src.evaluation.compare_runs --a {found[0]} --b {OUT} --out {OUT}/compare_with_{REFERENCE}.csv
else:
    print(f'no {REFERENCE}/per_image.csv in the inputs')

## 8. Save

*Save Version → Save & Run All*, then create a **Private** dataset from the output (for example `eval-smirk-em1`, or
`eval-<run>` for a trained model), or just download `summary.csv`. Section 7 of a later evaluation uses it as reference.